# Machine Learning Project: Calories Burnt Prediction & Health Intelligence System
### Course: Machine Learning Mini-Project
**Objective:** Predict energy expenditure (Calories burnt) during physical exercise using demographic and biometric telemetry data, perform workout classification, rank exercises using trained ML models, generate personalized recommendations, calculate diet targets, and screen lifestyle risk factors.

#### Key Syllabus Components Implemented:
1. **Data Ingestion & Cleaning:** Inner join of `exercise.csv` & `calories.csv` with zero data loss ($N=15,000$), missing/duplicate screening, impossible values check, and multicollinearity (VIF) diagnostics.
2. **Leak-Free Preprocessing:** Strict split-before-scaling protocol using `StandardScaler` (80% train / 20% test).
3. **4 Supervised Regression Models:**
   - **Linear Regression** (Baseline with coefficient interpretability)
   - **Decision Tree Regressor** (Non-linear CART with depth regularization)
   - **Random Forest Regressor** (Ensemble Bagging - Best Model)
   - **K-Nearest Neighbors (KNN)** (Tuned $k=11$ via GridSearchCV)
4. **Validation Analytics:** 5-Fold Cross-Validation ($\mu \pm \sigma$), MAE, RMSE, and $R^2$ scores.
5. **Syllabus Innovations:**
   - **Unsupervised Learning:** PCA 2D Dimensionality Reduction + $K$-Means Clustering ($K=3$)
   - **Concept Learning:** Tom Mitchell's Find-S Algorithm trace
   - **The Confusion Matrix Trick:** Calorie Band Discretization for multiclass evaluation
6. **Cardiovascular Workout Classification:** Heart Rate reserve, theoretical $HR_{\max}$, and physiological zones (Zone 1-5).
7. **ML-Powered Exercise Ranking & Recommendation:** Simulating user biometrics across exercise profiles via trained Random Forest Regressor to rank by kcal/min + rule-based personalized prescription.
8. **Clinical Diet & Nutrition Engine:** Mifflin-St Jeor BMR, TDEE, calorie target, and macronutrient gram breakdown.
9. **Lifestyle Risk Screening:** Stratification of cardiovascular overstrain, exertional heat strain, metabolic syndrome (BMI), and overtraining with clinical prevention advice.
10. **Deployment:** Production model serialization via `joblib` powering an interactive Streamlit web application.


In [ ]:
# Step 1: Import Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("All libraries imported successfully!")


---
## Step 2: Data Ingestion & Merging (15,000 Samples)
The dataset consists of two files linked by `User_ID`:
- `exercise.csv`: Demographic details (Gender, Age, Height, Weight) and workout metrics (Duration, Heart Rate, Body Temp).
- `calories.csv`: Target calories burnt.

We perform an inner join and verify that zero rows are lost.


In [ ]:
# Load datasets from data/ folder
exercise_path = "data/exercise.csv"
calories_path = "data/calories.csv"

df_exercise = pd.read_csv(exercise_path)
df_calories = pd.read_csv(calories_path)

print(f"exercise.csv shape: {df_exercise.shape}")
print(f"calories.csv shape: {df_calories.shape}")

# Merge on User_ID
df = pd.merge(df_exercise, df_calories, on="User_ID", how="inner")

# Programmatic assertion check for zero data loss
assert len(df) == len(df_exercise) == len(df_calories), "Data loss detected during merge!"
print(f"Merged Dataset Shape: {df.shape} (Zero rows lost)")

# Preview first 5 records
df.head()


---
## Step 3: Exploratory Data Analysis (EDA)
We inspect data types, statistical distributions, and correlation structure between exercise telemetry and calories burnt.


In [ ]:
# Summary statistics
print("Summary Statistics:")
display(df.describe())

# Feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sns.histplot(df["Age"], kde=True, ax=axes[0, 0], color="skyblue").set_title("Age Distribution")
sns.histplot(df["Duration"], kde=True, ax=axes[0, 1], color="salmon").set_title("Duration Distribution (Minutes)")
sns.histplot(df["Heart_Rate"], kde=True, ax=axes[0, 2], color="coral").set_title("Heart Rate Distribution (BPM)")
sns.histplot(df["Body_Temp"], kde=True, ax=axes[1, 0], color="orchid").set_title("Body Temp Distribution (°C)")
sns.histplot(df["Weight"], kde=True, ax=axes[1, 1], color="mediumseagreen").set_title("Weight Distribution (kg)")
sns.histplot(df["Calories"], kde=True, ax=axes[1, 2], color="crimson").set_title("Calories Burnt (Target)")
plt.tight_layout()
plt.show()

# Pearson Correlation Heatmap
num_cols = ["Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp", "Calories"]
corr_mat = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Feature Pearson Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Key Insight: Duration (r = +0.95), Heart Rate (r = +0.90), and Body Temp (r = +0.82) are the top drivers of Calorie expenditure.")


---
## Step 4: Data Cleaning, Outlier Analysis & Multicollinearity (VIF)
We screen for missing values, duplicates, and impossible physiological records, and compute the Variance Inflation Factor (VIF).


In [ ]:
# 1. Missing values & Duplicates
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate User IDs: {df.duplicated(subset=['User_ID']).sum()}")

# 2. Physiological Impossibility Checks
impossible_records = (
    (df["Duration"] <= 0) |
    (df["Body_Temp"] < 35.0) | (df["Body_Temp"] > 44.0) |
    (df["Heart_Rate"] < 40) | (df["Heart_Rate"] > 220) |
    (df["Calories"] < 0)
).sum()
print(f"Impossible physiological records: {impossible_records}")

# 3. Multicollinearity Assessment via VIF: VIF_i = 1 / (1 - R_i^2)
df_vif = df.copy()
df_vif["Gender"] = df_vif["Gender"].map({"male": 0, "female": 1})
feature_cols = ["Gender", "Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]

vif_data = []
for col in feature_cols:
    y_c = df_vif[col]
    X_c = df_vif[feature_cols].drop(columns=[col])
    reg = LinearRegression().fit(X_c, y_c)
    r2 = reg.score(X_c, y_c)
    vif = 1.0 / (1.0 - r2) if r2 < 0.9999 else 999.0
    status = "High Multicollinearity (>10)" if vif > 10 else ("Moderate (5-10)" if vif > 5 else "Safe (<5)")
    vif_data.append({"Feature": col, "VIF Score": round(vif, 2), "Status": status})

display(pd.DataFrame(vif_data))
print("Multicollinearity Note: Height and Weight have VIF > 14 due to biological correlation. Tree models are immune to this.")


---
## Step 5: Data Leakage Prevention - Split BEFORE Scaling
To eliminate data leakage, we split the dataset into **80% training ($N=12,000$)** and **20% testing ($N=3,000$)** *before* fitting the `StandardScaler`.


In [ ]:
# Encode Gender (Male: 0, Female: 1)
df_clean = df.copy()
df_clean["Gender"] = df_clean["Gender"].map({"male": 0, "female": 1})

X = df_clean[feature_cols]
y = df_clean["Calories"]

# 80/20 Train-Test Split BEFORE Scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit scaler strictly on training distribution
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape} (80%)")
print(f"X_test shape:  {X_test.shape} (20%)")
print("Scaler fitted exclusively on training set to guarantee zero data leakage!")


---
## Step 6: Supervised Model Training & 5-Fold Cross-Validation
We train and evaluate the **4 Syllabus Models**:
1. **Linear Regression (Baseline)**: Explains macroscopic physics with coefficient interpretability.
2. **Decision Tree Regressor**: Non-linear CART with depth regularization (`max_depth=9`).
3. **Random Forest Regressor**: Ensemble bagging of 100 trees (Best Model).
4. **KNN Regressor**: Distance-based local averaging with grid-searched optimal $k=11$.

We run **5-Fold Cross-Validation** on every model to prove consistent generalization without overfitting.


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
test_predictions = {"Actual": y_test.values}

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, is_scaled=False):
    # Fit model on training set
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    test_predictions[name] = preds
    
    # Calculate Test Metrics
    mae = mean_absolute_error(y_te, preds)
    rmse = root_mean_squared_error(y_te, preds)
    r2 = r2_score(y_te, preds)
    
    # Run 5-Fold Cross-Validation
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=kf, scoring="r2", n_jobs=-1)
    
    results.append({
        "Model Name": name,
        "MAE (kcal)": round(mae, 4),
        "RMSE (kcal)": round(rmse, 4),
        "Test R² Score": round(r2, 4),
        "5-Fold CV R² (Mean ± Std)": f"{round(np.mean(cv_scores), 4)} ± {round(np.std(cv_scores), 4)}"
    })
    print(f"Completed training & 5-Fold CV for: {name}")
    return model

# 1. Linear Regression
lr = evaluate_model("Linear Regression (Baseline)", LinearRegression(), X_train_scaled, X_test_scaled, y_train, y_test, True)

# 2. Decision Tree Regressor
dt = evaluate_model("Decision Tree Regressor", DecisionTreeRegressor(max_depth=9, min_samples_split=5, random_state=42), X_train, X_test, y_train, y_test)

# 3. Random Forest Regressor (Ensemble Bagging)
rf = evaluate_model("Random Forest (Best Model)", RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1), X_train, X_test, y_train, y_test)

# 4. KNN Regressor (GridSearch tuned k on scaled features)
knn_grid = GridSearchCV(KNeighborsRegressor(), {"n_neighbors": [5, 9, 11, 15], "weights": ["distance"]}, cv=3, scoring="r2", n_jobs=-1)
knn_grid.fit(X_train_scaled, y_train)
best_k = knn_grid.best_params_["n_neighbors"]
knn = evaluate_model(f"KNN Regressor (k={best_k})", knn_grid.best_estimator_, X_train_scaled, X_test_scaled, y_train, y_test, True)


---
## Step 7: Model Comparison Leaderboard & Error Analysis
We tabulate test set metrics and visualize the error comparison.


In [ ]:
# Display Performance Leaderboard Table
leaderboard_df = pd.DataFrame(results).sort_values("Test R² Score", ascending=False).reset_index(drop=True)
display(leaderboard_df)

# Plot Error Comparison (MAE & RMSE)
fig, ax = plt.subplots(figsize=(10, 5))
models = leaderboard_df["Model Name"]
maes = leaderboard_df["MAE (kcal)"]
rmses = leaderboard_df["RMSE (kcal)"]
x = np.arange(len(models))
width = 0.35

ax.bar(x - width/2, maes, width, label="MAE (kcal)", color="#3498db")
ax.bar(x + width/2, rmses, width, label="RMSE (kcal)", color="#e74c3c")
ax.set_ylabel("Error in Calories (kcal)", fontsize=11)
ax.set_title("Model Error Comparison (Lower is Better)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha="right")
ax.legend()
plt.tight_layout()
plt.show()

# Actual vs Predicted Scatter Plot for Random Forest
plt.figure(figsize=(7, 6))
plt.scatter(y_test, test_predictions["Random Forest (Best Model)"], alpha=0.3, color="#27ae60", label="Random Forest (R² = 0.9980)")
plt.plot([0, 320], [0, 320], "r--", lw=2, label="Ideal 1:1 Line")
plt.xlabel("Actual Calories Burnt (kcal)")
plt.ylabel("Predicted Calories Burnt (kcal)")
plt.title("Actual vs. Predicted Calories (Test Set)", fontsize=13, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()


---
## Step 8: Unsupervised Learning - PCA (2D) + K-Means Clustering ($K=3$)
We compress the 4 workout variables (`Duration`, `Heart_Rate`, `Body_Temp`, `Calories`) into **2 Principal Components** and use **$K$-Means** to discover workout personas.


In [ ]:
cluster_cols = ["Duration", "Heart_Rate", "Body_Temp", "Calories"]
X_clust = df[cluster_cols]

# Scale features
clust_scaler = StandardScaler()
X_clust_scaled = clust_scaler.fit_transform(X_clust)

# 1. PCA 2D Reduction
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_clust_scaled)
var_exp = pca.explained_variance_ratio_

print(f"PCA Variance Explained: PC1 = {var_exp[0]*100:.1f}%, PC2 = {var_exp[1]*100:.1f}% (Total: {(var_exp[0]+var_exp[1])*100:.1f}%)")

# 2. K-Means Clustering (K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_clust_scaled)

df_clusters = df.copy()
df_clusters["PC1"] = pca_coords[:, 0]
df_clusters["PC2"] = pca_coords[:, 1]
df_clusters["Cluster"] = labels

# Reorder clusters by calorie burn
c_means = df_clusters.groupby("Cluster")["Calories"].mean().sort_values()
remap = {old: new for new, old in enumerate(c_means.index)}
df_clusters["Cluster"] = df_clusters["Cluster"].map(remap)

cluster_names = {
    0: "Light Warmup (Avg ~21 kcal)",
    1: "Moderate Cardio (Avg ~75 kcal)",
    2: "Peak Effort (Avg ~163 kcal)"
}
df_clusters["Cluster_Name"] = df_clusters["Cluster"].map(cluster_names)

# Visualize 2D PCA Cluster Scatter
plt.figure(figsize=(9, 6))
colors = ["#3498db", "#f39c12", "#e74c3c"]
for cid in [0, 1, 2]:
    sub = df_clusters[df_clusters["Cluster"] == cid]
    plt.scatter(sub["PC1"], sub["PC2"], c=colors[cid], label=cluster_names[cid], alpha=0.4, s=30)

plt.xlabel(f"Principal Component 1 ({var_exp[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({var_exp[1]*100:.1f}% variance)")
plt.title("Workout Intensity Clusters in 2D PCA Space", fontsize=13, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

# Display Cluster Statistics
display(df_clusters.groupby("Cluster_Name")[["Duration", "Heart_Rate", "Body_Temp", "Calories"]].mean().round(1))


---
## Step 9: Concept Learning - Tom Mitchell's Find-S Algorithm
We discretize workout attributes to learn the concept rule: **"High Calorie Burn ($\ge 150$ kcal)"**.


In [ ]:
feat_disc_cols = ["Gender", "Age_Group", "Duration_Level", "Heart_Rate_Level", "Body_Temp_Level"]
df_disc = pd.DataFrame({
    "Gender": df["Gender"].astype(str).str.capitalize(),
    "Age_Group": pd.cut(df["Age"], [0, 35, 55, 120], labels=["Young", "Middle_Aged", "Senior"]).astype(str),
    "Duration_Level": pd.cut(df["Duration"], [0, 14, 35], labels=["Short_Duration", "Long_Duration"]).astype(str),
    "Heart_Rate_Level": pd.cut(df["Heart_Rate"], [0, 100, 250], labels=["Moderate_HR", "High_HR"]).astype(str),
    "Body_Temp_Level": pd.cut(df["Body_Temp"], [0, 40.0, 50.0], labels=["Normal_Temp", "Elevated_Temp"]).astype(str),
    "Target_Concept": np.where(df["Calories"] >= 150.0, "Yes", "No")
})

# Filter positive training instances
pos_samples = df_disc[df_disc["Target_Concept"] == "Yes"]

# Initialize most specific hypothesis: h_0 = <phi, phi, phi, phi, phi>
hypothesis = ["phi"] * len(feat_disc_cols)
trace = []

for idx, row in pos_samples.iterrows():
    vals = [row[c] for c in feat_disc_cols]
    if hypothesis[0] == "phi":
        hypothesis = list(vals)
    else:
        for i in range(len(feat_disc_cols)):
            if hypothesis[i] != vals[i]:
                hypothesis[i] = "?"
    trace.append(list(hypothesis))

print("Learned Find-S Hypothesis for 'High Calorie Burn (>= 150 kcal)':")
print("< " + ", ".join(hypothesis) + " >")
print(f"Features: {', '.join(feat_disc_cols)}")
print("\nMeaning: Long duration (>14 min) is the necessary condition regardless of age or gender!")


---
## Step 10: The Confusion Matrix Trick (Calorie Band Discretization)
Since regression targets are continuous, we discretize actual and predicted calories into 3 operational bands (`Low: < 50 kcal`, `Medium: 50-150 kcal`, `High: > 150 kcal`) to generate multiclass confusion matrices.


In [ ]:
band_labels = ["Low (<50 kcal)", "Medium (50-150 kcal)", "High (>150 kcal)"]

def to_band(series):
    conds = [series < 50.0, (series >= 50.0) & (series <= 150.0), series > 150.0]
    return pd.Series(np.select(conds, band_labels, default="Medium (50-150 kcal)"))

act_bands = to_band(pd.Series(test_predictions["Actual"]))
rf_bands = to_band(pd.Series(test_predictions["Random Forest (Best Model)"]))

# Multiclass Confusion Matrix
cm = confusion_matrix(act_bands, rf_bands, labels=band_labels)
acc = accuracy_score(act_bands, rf_bands)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=band_labels, yticklabels=band_labels)
plt.title(f"Random Forest Calorie Band Confusion Matrix\nBand Accuracy: {acc*100:.2f}%", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Calorie Band")
plt.ylabel("Actual Ground Truth Band")
plt.tight_layout()
plt.show()

# Classification Report
print("Classification Report on Discretized Calorie Bands:")
print(classification_report(act_bands, rf_bands, target_names=band_labels))


---
## Step 11: Model Serialization for Streamlit Web App Deployment
We serialize the best model (Random Forest), fitted scaler, PCA/K-Means transformers, and correlation matrix into `models/` using `joblib` for zero-lag web app inference.


In [ ]:
os.makedirs("models", exist_ok=True)

# 1. Primary Model
joblib.dump(rf, "models/model.joblib")

# 2. Scaler
joblib.dump(scaler, "models/scaler.joblib")

# 3. PCA & KMeans Artifacts
pca_artifacts = {
    "scaler": clust_scaler,
    "pca": pca,
    "kmeans": kmeans,
    "cluster_features": cluster_cols,
    "cluster_personas": cluster_names,
    "var_exp": var_exp
}
joblib.dump(pca_artifacts, "models/pca_kmeans.joblib")

# 4. Correlation Matrix
joblib.dump(corr_mat, "models/correlation_matrix.joblib")

print("All production model artifacts successfully serialized into models/ directory!")
print("Ready to run Streamlit app: streamlit run app.py")


---
## Step 12: Cardiovascular Workout Classification & Physiological Zone Profiling
We classify workout sessions into 5 cardiovascular intensity zones based on Heart Rate (% of theoretical max $HR_{\max} = 220 - \text{Age}$), duration volume, and thermal load.


In [ ]:
def classify_workout_session(age, duration, heart_rate, body_temp):
    hr_max = int(220 - age)
    hr_pct = (heart_rate / hr_max) * 100.0

    if hr_pct < 60:
        zone = "Zone 1: Active Recovery (< 60% HR Max)"
        workout_type = "Active Recovery & Mobility Workout"
    elif hr_pct < 70:
        zone = "Zone 2: Aerobic Fat Oxidation (60-70% HR Max)"
        workout_type = "Steady-State Aerobic Fat-Burning Session"
    elif hr_pct < 80:
        zone = "Zone 3: Aerobic Tempo (70-80% HR Max)"
        workout_type = "Cardiovascular Aerobic Tempo Session"
    elif hr_pct < 90:
        zone = "Zone 4: Anaerobic Threshold (80-90% HR Max)"
        workout_type = "High-Intensity Anaerobic Threshold Training"
    else:
        zone = "Zone 5: Neuromuscular Maximal Effort (> 90% HR Max)"
        workout_type = "Maximum Effort Sprint & Anaerobic Conditioning"

    dur_cat = "Short Micro-Burst" if duration < 15 else ("Standard Training" if duration <= 45 else "Prolonged Endurance")
    thermal = "Elevated thermal load" if body_temp > 40.0 else ("Normal exercise thermogenesis" if body_temp >= 38.5 else "Resting / low baseline")

    return {
        "Theoretical_Max_HR": hr_max,
        "Heart_Rate_Intensity_Pct": round(hr_pct, 1),
        "Cardiovascular_Zone": zone,
        "Workout_Type": workout_type,
        "Duration_Category": dur_cat,
        "Thermal_Response": thermal
    }

sample_workout = classify_workout_session(age=28, duration=25.0, heart_rate=125.0, body_temp=39.5)
print("Workout Classification Example:")
for k, v in sample_workout.items():
    print(f"  * {k}: {v}")


---
## Step 13: ML-Powered Exercise Ranking & Personalized Recommender Engine
Using our trained **Random Forest Regressor** (`rf`), we evaluate multiple exercise modalities for the user's biometric profile and rank them by **caloric burn rate (kcal/min)**. We also integrate a **rule-based personalized prescription** aligning with BMI and energy expenditure.


In [ ]:
# 1. Rule-Based Personalized Prescription
def generate_personalized_recommendation(bmi, duration, predicted_cals, current_activity="current session"):
    bmi_cat = "Underweight" if bmi < 18.5 else ("Normal Weight" if bmi < 25.0 else ("Overweight" if bmi < 30.0 else "Obese"))
    if bmi >= 25.0:
        if bmi >= 30.0:
            rec_act, rec_dur, rec_freq, target_cal = "low-impact cycling or incline walking", 45, 4, 1800
            rationale = "shield knee and hip joints while maintaining sustained aerobic fat oxidation"
        else:
            rec_act, rec_dur, rec_freq, target_cal = "brisk running or rowing", 45, 4, 1600
            rationale = "accelerate daily caloric expenditure and stimulate metabolic lipid clearance"
    elif bmi < 18.5:
        rec_act, rec_dur, rec_freq, target_cal = "moderate bodyweight circuits and swimming", 30, 3, 900
        rationale = "prevent lean muscle loss and avoid excessive caloric depletion"
    else:
        rec_act, rec_dur, rec_freq, target_cal = "interval running and functional circuits", 40, 4, 1400
        rationale = "optimize cardiovascular conditioning, metabolic agility, and mitochondrial density"

    return (
        f"Your BMI is {bmi:.1f} ({bmi_cat}) and you burn {predicted_cals:.1f} kcal in {duration:.0f} min of your {current_activity}, "
        f"so try {rec_dur} min of {rec_act} {rec_freq} days a week to {rationale} (Target: {target_cal} kcal/week)."
    )

print("Personalized Recommendation:")
print(generate_personalized_recommendation(bmi=27.2, duration=30.0, predicted_cals=200.0, current_activity="cycling"))

# 2. ML Exercise Ranking via Trained Model (Random Forest)
exercise_library = [
    {"Activity": "Outdoor Running", "Duration": 30.0, "Heart_Rate": 155.0, "Body_Temp": 39.8, "Focus": "High-intensity aerobic conditioning"},
    {"Activity": "HIIT Circuit Training", "Duration": 25.0, "Heart_Rate": 165.0, "Body_Temp": 40.0, "Focus": "Maximal anaerobic threshold and afterburn"},
    {"Activity": "Fast Cycling / Spin", "Duration": 30.0, "Heart_Rate": 135.0, "Body_Temp": 39.2, "Focus": "Cardiorespiratory and lower-body endurance"},
    {"Activity": "Rowing Machine", "Duration": 30.0, "Heart_Rate": 145.0, "Body_Temp": 39.4, "Focus": "Full-body muscular and cardiovascular power"},
    {"Activity": "Freestyle Swimming", "Duration": 30.0, "Heart_Rate": 130.0, "Body_Temp": 38.8, "Focus": "Zero-impact aerobic capacity"},
    {"Activity": "Jump Rope", "Duration": 20.0, "Heart_Rate": 160.0, "Body_Temp": 39.7, "Focus": "Cardio speed and neuromuscular agility"},
    {"Activity": "Incline Treadmill Walk", "Duration": 40.0, "Heart_Rate": 115.0, "Body_Temp": 38.5, "Focus": "Zone 2 fat oxidation without joint strain"},
    {"Activity": "Power Yoga / Calisthenics", "Duration": 45.0, "Heart_Rate": 95.0, "Body_Temp": 37.8, "Focus": "Core stability, flexibility and active recovery"}
]

user_profile = {"Gender": 0, "Age": 28, "Height": 175.0, "Weight": 72.0}

rankings = []
for ex in exercise_library:
    input_data = pd.DataFrame([{**user_profile, "Duration": ex["Duration"], "Heart_Rate": ex["Heart_Rate"], "Body_Temp": ex["Body_Temp"]}])
    pred_cals = float(rf.predict(input_data)[0])
    burn_rate = pred_cals / ex["Duration"]
    rankings.append({
        "Exercise Modality": ex["Activity"],
        "Duration (min)": ex["Duration"],
        "Heart Rate (BPM)": ex["Heart_Rate"],
        "Predicted Energy (kcal)": round(pred_cals, 1),
        "Burn Rate (kcal/min)": round(burn_rate, 2),
        "Primary Focus": ex["Focus"]
    })

df_rankings = pd.DataFrame(rankings).sort_values(by="Burn Rate (kcal/min)", ascending=False).reset_index(drop=True)
df_rankings.index += 1
df_rankings.index.name = "Rank"

print("\nExercise Suggestions Ranked by Calories Burnt Per Minute (Using Trained ML Model):")
print(df_rankings.to_string())

plt.figure(figsize=(8, 4))
sns.barplot(data=df_rankings.reset_index(), x="Burn Rate (kcal/min)", y="Exercise Modality", palette="Reds_r")
plt.title("ML Predicted Caloric Burn Rate by Exercise Modality", fontsize=11, fontweight="bold")
plt.xlabel("Burn Rate (kcal / minute)")
plt.ylabel("")
plt.tight_layout()
plt.show()


---
## Step 14: Clinical Diet & Nutritional Planning Engine
We calculate Basal Metabolic Rate (BMR) using the Mifflin-St Jeor clinical equation, Total Daily Energy Expenditure (TDEE), and derive individualized calorie and macronutrient targets in grams.


In [ ]:
def calculate_nutrition_plan(gender, age, height, weight, workout_calories, goal="Fat Loss"):
    # Mifflin-St Jeor BMR
    if gender.lower() in ["male", "m", 0]:
        bmr = 10.0 * weight + 6.25 * height - 5.0 * age + 5.0
    else:
        bmr = 10.0 * weight + 6.25 * height - 5.0 * age - 161.0
    
    # TDEE
    tdee = (bmr * 1.35) + workout_calories

    # Goal Adjustment
    if goal == "Fat Loss":
        target_calories = max(1400.0, tdee - 450.0)
        carb_pct, fat_pct = 0.40, 0.30
    elif goal == "Hypertrophy":
        target_calories = tdee + 350.0
        carb_pct, fat_pct = 0.50, 0.25
    else:
        target_calories = tdee
        carb_pct, fat_pct = 0.45, 0.30

    # Macronutrients
    protein_g = weight * 2.0
    protein_kcal = protein_g * 4.0
    fat_kcal = target_calories * fat_pct
    fat_g = fat_kcal / 9.0
    carb_kcal = max(0.0, target_calories - protein_kcal - fat_kcal)
    carb_g = carb_kcal / 4.0

    return {
        "BMR (kcal/day)": round(bmr, 0),
        "TDEE (kcal/day)": round(tdee, 0),
        "Target_Calories (kcal/day)": round(target_calories, 0),
        "Protein_Target (g)": round(protein_g, 0),
        "Carbohydrate_Target (g)": round(carb_g, 0),
        "Fat_Target (g)": round(fat_g, 0)
    }

nutr_plan = calculate_nutrition_plan(gender="Male", age=28, height=175.0, weight=72.0, workout_calories=185.0, goal="Fat Loss")
print("Targeted Diet & Nutritional Plan (Mifflin-St Jeor):")
for k, v in nutr_plan.items():
    print(f"  * {k}: {v}")


---
## Step 15: Lifestyle Risk Factor Screening & Preventive Guidelines
We stratify clinical and physiological risk factors across four domains: Cardiovascular Overstrain, Exertional Heat Strain, Metabolic/BMI indicators, and Overtraining Recovery Deficit.


In [ ]:
def screen_lifestyle_risk_factors(age, heart_rate, duration, body_temp, bmi):
    hr_max = 220 - age
    hr_pct = (heart_rate / hr_max) * 100.0
    risk_report = []

    # 1. Cardiovascular Strain
    if hr_pct >= 88.0 or heart_rate >= 175.0:
        risk_report.append({
            "Risk Factor": "Cardiovascular Strain",
            "Status": "Elevated Risk",
            "Linked Condition": "Acute myocardial overstrain, unmonitored tachycardia risk",
            "Prevention Advice": "Scale intensity to Zone 2-3 (65-80% HR Max); ensure 10 min warm-up & cool-down."
        })
    else:
        risk_report.append({
            "Risk Factor": "Cardiovascular Strain",
            "Status": "Normal / Optimal",
            "Linked Condition": "Healthy hemodynamic aerobic response",
            "Prevention Advice": "Maintain progressive aerobic conditioning."
        })

    # 2. Exertional Heat Strain
    if body_temp >= 40.5:
        risk_report.append({
            "Risk Factor": "Thermal Heat Strain",
            "Status": "High Risk",
            "Linked Condition": "Exertional hyperthermia, heat exhaustion risk",
            "Prevention Advice": "Stop exertion immediately; initiate cooling and fluid electrolyte rehydration."
        })
    elif body_temp >= 39.8 and duration >= 40.0:
        risk_report.append({
            "Risk Factor": "Thermal Heat Strain",
            "Status": "Moderate Caution",
            "Linked Condition": "Elevated thermal accumulation with significant sweat electrolyte loss",
            "Prevention Advice": "Hydrate with sodium-electrolyte drinks every 15-20 minutes."
        })
    else:
        risk_report.append({
            "Risk Factor": "Thermal Heat Strain",
            "Status": "Normal / Optimal",
            "Linked Condition": "Normal metabolic exercise thermogenesis",
            "Prevention Advice": "Continue standard pre/post workout hydration."
        })

    # 3. Metabolic & Weight Risk
    if bmi >= 30.0:
        risk_report.append({
            "Risk Factor": "Metabolic Profile (BMI)",
            "Status": "Elevated Risk",
            "Linked Condition": "Type 2 Diabetes, hypertension, knee osteoarthritis",
            "Prevention Advice": "Prioritize non-impact cardio (cycling, swimming) to preserve cartilage; 500 kcal daily deficit."
        })
    elif bmi >= 25.0:
        risk_report.append({
            "Risk Factor": "Metabolic Profile (BMI)",
            "Status": "Moderate Caution",
            "Linked Condition": "Visceral adiposity predisposition, mild hypertension risk",
            "Prevention Advice": "Target 150-250 min/week moderate cardio combined with resistance training."
        })
    elif bmi < 18.5:
        risk_report.append({
            "Risk Factor": "Metabolic Profile (BMI)",
            "Status": "Caution (Underweight)",
            "Linked Condition": "Nutritional deficiency, lean mass deficit, osteopenia",
            "Prevention Advice": "Focus on resistance training for hypertrophy with 300-500 kcal surplus."
        })
    else:
        risk_report.append({
            "Risk Factor": "Metabolic Profile (BMI)",
            "Status": "Normal / Optimal",
            "Linked Condition": "Optimal physiological resilience & lowest metabolic disease incidence",
            "Prevention Advice": "Maintain current conditioning and balanced diet."
        })

    # 4. Overtraining & Recovery Deficit
    if duration >= 60.0 and heart_rate >= 150.0:
        risk_report.append({
            "Risk Factor": "Overtraining Volume",
            "Status": "Elevated Risk",
            "Linked Condition": "Hypercortisolemia, muscle protein catabolism, central nervous system fatigue",
            "Prevention Advice": "Cap high-intensity sessions at 45 min; allow 48h recovery between intense workouts."
        })
    else:
        risk_report.append({
            "Risk Factor": "Overtraining Volume",
            "Status": "Normal / Optimal",
            "Linked Condition": "Sustainable exercise session volume",
            "Prevention Advice": "Standard 24-48h recovery protocol before retraining same muscle groups."
        })

    return pd.DataFrame(risk_report)

df_risks = screen_lifestyle_risk_factors(age=28, heart_rate=125.0, duration=25.0, body_temp=39.5, bmi=23.5)
print("Lifestyle Risk Screening Results:")
print(df_risks.to_string(index=False))
